# Temperature Yaklaşımlar (Top-k, Top-p)

In [ ]:
import torch
import torch.nn.functional as F
import numpy as np

# --- GİRİŞ VE KURULUM ---
# Bu script, bir dil modelinin metin üretme anındaki tek bir adımını simüle eder.
# Amaç, Temperature, Greedy Search, Top-k ve Top-p (Nucleus) gibi kavramların
# modelin nihai çıktısını nasıl şekillendirdiğini adım adım görmektir.
#
# Senaryo: Bir dil modeli, "İstanbul Türkiye'nin en kalabalık" cümlesini işledi
# ve şimdi bir sonraki kelimeye karar verecek. Modelin olası kelimeler için
# ürettiği ham skorlardan (logit) yola çıkarak, farklı parametrelerin sonucu
# nasıl değiştirdiğini inceleyeceğiz.

print("--- Adım Adım Üretim Simülasyonu ---")
print("Senaryo: Model, 'İstanbul Türkiye'nin en kalabalık' girdisini işledi.")
print("Şimdi bir sonraki kelimeye karar verecek.\n")

# Modelin ürettiği ham skorları (logit'leri) temsil eden bir tensor oluşturalım.
# Yüksek değer, o kelimenin daha olası olduğunu gösterir.
vocab = ["şehri", "bölgesi", "güzeli", "yeri", "yemeğidir"]
logits = torch.tensor([4.5, 2.5, 2.0, 1.0, -1.0])

print("1. Adım: Modelin Ham Çıktısı (Logits)")
for word, logit in zip(vocab, logits):
    print(f"- {word:<10}: {logit.item():.2f}")
print("-" * 40)


# --- BÖLÜM 1: TEMPERATURE (ISI) KAVRAMININ ETKİSİ ---
print("\n### BÖLÜM 1: Temperature Değerinin Olasılıklara Etkisi ###\n")

def calculate_probs(logits, temperature=1.0):
    """Verilen logit ve sıcaklık değeri için olasılıkları hesaplar ve yazdırır."""
    print(f"--- Sıcaklık (Temperature) T = {temperature} ---")
    
    # Sıcaklık değeri, Softmax fonksiyonuna girmeden önce logit'leri ölçeklendirir.
    scaled_logits = logits / temperature
    print("Sıcaklıkla ölçeklenmiş logitler:")
    for word, logit in zip(vocab, scaled_logits):
        print(f"- {word:<10}: {logit.item():.2f}")
        
    # Ölçeklenmiş logit'lerden olasılıkları hesapla (Softmax)
    probabilities = F.softmax(scaled_logits, dim=0)
    
    print("\nSonuç Olasılık Dağılımı:")
    for word, prob in zip(vocab, probabilities):
        print(f"- {word:<10}: %{prob.item()*100:.2f}")
    print("-" * 40)
    return probabilities

# Farklı sıcaklık değerleri ile olasılıkları hesaplayalım
probs_normal_temp = calculate_probs(logits, temperature=1.0) # Normal, referans durum
probs_low_temp = calculate_probs(logits, temperature=0.5)   # Düşük ısı: Model daha emin ve odaklı
probs_high_temp = calculate_probs(logits, temperature=1.5)  # Yüksek ısı: Model daha rastgele ve yaratıcı


# --- BÖLÜM 2: ÖRNEKLEME (SAMPLING) STRATEJİLERİ ---
print("\n### BÖLÜM 2: Olasılık Dağılımından Kelime Seçme Stratejileri ###")
print("Bu bölüm için T=1.0 ile üretilen normal olasılık dağılımını kullanacağız.\n")

# Strateji 1: Greedy Search (Açgözlü Arama)
print("--- Strateji 1: Greedy Search ---")
print("Yöntem: Her zaman en yüksek olasılığa sahip kelimeyi seç.")
greedy_index = torch.argmax(probs_normal_temp)
greedy_word = vocab[greedy_index]
print(f"Seçilen Kelime: '{greedy_word}' (Olasılık: %{probs_normal_temp[greedy_index].item()*100:.2f})\n")

# Strateji 2: Top-k Sampling
print("--- Strateji 2: Top-k Sampling (k=3) ---")
print("Yöntem: En olası 'k' adet kelimeden oluşan bir havuz oluştur ve bu havuzdan rastgele seç.")
k = 3
top_k_probs, top_k_indices = torch.topk(probs_normal_temp, k)

print("Oluşturulan 'Top 3' havuzu:")
for i in range(k):
    word = vocab[top_k_indices[i]]
    prob = top_k_probs[i]
    print(f"- {word:<10}: %{prob.item()*100:.2f}")

# Seçimi bu daraltılmış havuzdan yap
# torch.multinomial, verilen olasılıklara göre bir örnek seçer.
top_k_sampled_index_in_pool = torch.multinomial(top_k_probs, num_samples=1)
# Havuzdaki indeksi, orijinal kelime haznesindeki indekse çevir
final_top_k_index = top_k_indices[top_k_sampled_index_in_pool]
top_k_word = vocab[final_top_k_index]

print(f"\nBu havuzdan rastgele seçilen kelime: '{top_k_word}'\n")

# Strateji 3: Top-p (Nucleus) Sampling
print("--- Strateji 3: Top-p (Nucleus) Sampling (p=0.90) ---")
print("Yöntem: Kümülatif olasılığı 'p' değerini geçen en küçük kelime setini oluştur ve oradan seç.")
p = 0.90

# Olasılıkları büyükten küçüğe sırala
sorted_probs, sorted_indices = torch.sort(probs_normal_temp, descending=True)
cumulative_probs = torch.cumsum(sorted_probs, dim=-1)

# Kümülatif toplamı p'yi geçenleri havuzun dışında bırak
indices_to_remove = cumulative_probs > p
# Ancak ilk kelimeyi her zaman tutmak için ilk indeksi False yap
indices_to_remove[..., 1:] = indices_to_remove[..., :-1].clone()
indices_to_remove[..., 0] = 0

indices_of_words_to_keep = sorted_indices[~indices_to_remove]
final_probs_for_p_sampling = sorted_probs[~indices_to_remove]

print("Oluşturulan 'Nucleus (Çekirdek)' havuzu (Toplam olasılık >= %90):")
for i in range(len(indices_of_words_to_keep)):
    word = vocab[indices_of_words_to_keep[i]]
    prob = probs_normal_temp[indices_of_words_to_keep[i]]
    print(f"- {word:<10}: %{prob.item()*100:.2f}")

# Seçimi bu dinamik havuzdan yap
top_p_sampled_index_in_pool = torch.multinomial(final_probs_for_p_sampling, num_samples=1)
final_top_p_index = indices_of_words_to_keep[top_p_sampled_index_in_pool]
top_p_word = vocab[final_top_p_index]
print(f"\nBu havuzdan rastgele seçilen kelime: '{top_p_word}'")
print("-" * 40)